# Nemotron SFT **v26** — SpatialClaw-style *verify-revise* traces (hard categories)

Inspired by NVIDIA's **SpatialClaw** ("the action interface matters — Code → Inspect → Revise
beats single-pass guessing"). We can't use tools at eval (greedy, `\boxed{}` only), so we move
the idea where it's legal: a **Python kernel generates the training traces**.

For each hard problem the kernel **executes every candidate rule against ALL given examples**,
**inspects** the match, and **revises** to the next hypothesis until one reproduces every example
— then applies it to the query. That search log *is* the chain-of-thought:

```
<think>
- Hypothesis: concatenation. 21>90 -> 2190, example 31. reject, revise.
- Hypothesis: addition. 21>90 -> 111, example 31. reject, revise.
  ... (rejects more)
- Hypothesis: digit-wise subtract mod 10. Verify all: 21>90->31 OK, 74>99->85 OK ... all match.
Rule = digit-wise subtract mod 10. Apply 24>28 -> 06.
</think>
\boxed{06}
```

**Why this wins the hard cats** (cryptarithm 8%, equation_guess): the deterministic solver fails
there because the model *guesses one rule and commits*. These traces teach it to **search the rule
space and self-verify** — the exact behaviour those categories require. Labels are perfect (the
kernel only emits a problem whose accepted rule reproduces every example and the query answer).

Self-contained: the candidate-op library is vendored inline (no `syn_datagen` upload). Output is
**SFT-ready** (`generated_cot` CSV + chat JSONL) — feed it to v24 (SFT) or v25 (RAFT) on the 0.85
checkpoint. Deterministic, infinite, zero API cost.


In [ ]:
import os

SEED      = 42
PER_CAT   = 400          # verify-revise traces per hard category
CATEGORIES = ["equation_numeric_deduce", "equation_numeric_guess",
              "cryptarithm_deduce", "cryptarithm_guess"]

OUT_DIR   = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_CSV   = os.path.join(OUT_DIR, "spatialclaw_verify_revise.csv")     # drop-in for v24 SFT loader
OUT_JSONL = os.path.join(OUT_DIR, "spatialclaw_verify_revise.jsonl")   # chat format (SFT/RAFT)

# Same compact system prompt as v24/v25 — but it tells the model to SEARCH + verify.
SYSTEM_PROMPT = (
    "You solve deterministic symbol/equation puzzles. Find the secret rule by TESTING candidate "
    "operations (concatenation, +, -, *, digit-wise ops, etc.) against EVERY given example and "
    "rejecting any that fails; then apply the surviving rule to the query. Put all reasoning in "
    "one <think> ... </think> block, then output the answer once as \\boxed{...} with nothing after it."
)
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({"PER_CAT": PER_CAT, "cats": len(CATEGORIES), "out": OUT_CSV})


In [ ]:
"""SpatialClaw verify-revise engine v2: rule SEARCH (Code->Inspect->Revise) +
EXPLICIT ARITHMETIC in every verification and the query, so a small LoRA learns to
compute, not memorize. Deterministic, perfect labels, no API."""
import random, re

# ── candidate-op library (vendored from syn_datagen/equation_numeric.py) ──
def _common_candidates(a, b, sa, sb):
    return [("concatenation", sa + sb), ("reverse concatenation", sb + sa),
            ("addition", str(a + b)), ("absolute difference", str(abs(a - b))),
            ("negated absolute difference", str(-abs(a - b))),
            ("subtraction (a-b)", str(a - b)), ("reverse subtraction (b-a)", str(b - a)),
            ("multiplication", str(a * b))]

def _rare_candidates(a, b, sa, sb):
    out = []
    if b != 0:
        out += [("integer division (a/b)", str(a // b)), ("modulo (a mod b)", str(a % b))]
    if len(sa) == 2 and len(sb) == 2:
        d1, d2, d3, d4 = int(sa[0]), int(sa[1]), int(sb[0]), int(sb[1])
        det = d1 * d4 - d2 * d3
        out += [("digit add mod10", str((d1 + d3) % 10) + str((d2 + d4) % 10)),
                ("digit sub mod10", str((d1 - d3) % 10) + str((d2 - d4) % 10)),
                ("cross multiply", str(d1 * d3 + d2 * d4)),
                ("digit multiply", str(d1 * d3) + str(d2 * d4)),
                ("determinant", str(det)), ("abs determinant", str(abs(det)))]
    return out

def _cand_map(a, b):
    return dict(_common_candidates(a, b, str(a), str(b)) + _rare_candidates(a, b, str(a), str(b)))

DESC = {"concatenation": "concatenation (write a then b)",
        "reverse concatenation": "reverse concatenation (write b then a)",
        "addition": "addition (a + b)", "absolute difference": "absolute difference |a - b|",
        "negated absolute difference": "negated absolute difference -|a - b|",
        "subtraction (a-b)": "subtraction (a - b)", "reverse subtraction (b-a)": "reverse subtraction (b - a)",
        "multiplication": "multiplication (a * b)", "integer division (a/b)": "integer division (a // b)",
        "modulo (a mod b)": "modulo (a mod b)", "digit add mod10": "digit-wise add mod 10",
        "digit sub mod10": "digit-wise subtract mod 10", "digit multiply": "digit-wise multiply",
        "cross multiply": "cross multiply", "determinant": "determinant", "abs determinant": "absolute determinant"}
COMMON = ["concatenation", "reverse concatenation", "addition", "absolute difference",
          "negated absolute difference", "subtraction (a-b)", "reverse subtraction (b-a)", "multiplication"]
RARE = ["integer division (a/b)", "modulo (a mod b)", "digit add mod10", "digit sub mod10",
        "cross multiply", "digit multiply", "determinant", "abs determinant"]

# ── explicit-arithmetic renderers (show the WORK) ──
_T = lambda n: (n // 10) * 10
_U = lambda n: n % 10

def arith(op, a, b):
    sa, sb = str(a), str(b)
    d1, d2, d3, d4 = int(sa[0]), int(sa[1]), int(sb[0]), int(sb[1])
    if op == "concatenation": return f"write digits of {a} then {b} = {sa}{sb}"
    if op == "reverse concatenation": return f"write digits of {b} then {a} = {sb}{sa}"
    if op == "addition": return f"{a}+{b} = ({_T(a)}+{_T(b)}) + ({_U(a)}+{_U(b)}) = {_T(a)+_T(b)} + {_U(a)+_U(b)} = {a+b}"
    if op == "subtraction (a-b)": return f"{a}-{b} = ({_T(a)}-{_T(b)}) + ({_U(a)}-{_U(b)}) = {_T(a)-_T(b)} + {_U(a)-_U(b)} = {a-b}"
    if op == "reverse subtraction (b-a)": return f"{b}-{a} = ({_T(b)}-{_T(a)}) + ({_U(b)}-{_U(a)}) = {_T(b)-_T(a)} + {_U(b)-_U(a)} = {b-a}"
    if op == "absolute difference": return f"|{a}-{b}| = |{a-b}| = {abs(a-b)}"
    if op == "negated absolute difference": return f"-|{a}-{b}| = -{abs(a-b)} = {-abs(a-b)}"
    if op == "multiplication": return f"{a}*{b} = {a}*{_T(b)} + {a}*{_U(b)} = {a*_T(b)} + {a*_U(b)} = {a*b}"
    if op == "integer division (a/b)": return f"{a} // {b} = {a//b}  (since {b}*{a//b} = {b*(a//b)} <= {a})"
    if op == "modulo (a mod b)": return f"{a} mod {b} = {a} - {b}*{a//b} = {a} - {b*(a//b)} = {a%b}"
    if op == "digit add mod10": return f"({d1}+{d3}) mod 10 = {(d1+d3)%10}, ({d2}+{d4}) mod 10 = {(d2+d4)%10} -> {(d1+d3)%10}{(d2+d4)%10}"
    if op == "digit sub mod10": return f"({d1}-{d3}) mod 10 = {(d1-d3)%10}, ({d2}-{d4}) mod 10 = {(d2-d4)%10} -> {(d1-d3)%10}{(d2-d4)%10}"
    if op == "digit multiply": return f"{d1}*{d3} = {d1*d3}, {d2}*{d4} = {d2*d4} -> {d1*d3}{d2*d4}"
    if op == "cross multiply": return f"{d1}*{d3} + {d2}*{d4} = {d1*d3} + {d2*d4} = {d1*d3+d2*d4}"
    if op == "determinant": return f"{d1}*{d4} - {d2}*{d3} = {d1*d4} - {d2*d3} = {d1*d4-d2*d3}"
    if op == "abs determinant": return f"|{d1}*{d4} - {d2}*{d3}| = |{d1*d4-d2*d3}| = {abs(d1*d4-d2*d3)}"
    return f"(op {op})"

def search_equation(examples):
    """Return (accepted_op, reject_log, checks): first op matching ALL examples."""
    reject = []
    for name in COMMON + RARE:
        checks = [(a, b, _cand_map(a, b).get(name), res) for (a, b, res) in examples]
        bad = [(a, b, p, r) for (a, b, p, r) in checks if p != r]
        if not bad:
            return name, reject, checks
        reject.append((name, bad[0]))
    return None, reject, None

def render_equation(examples, sym, qa, qb):
    op, reject, checks = search_equation(examples)
    if op is None:
        return None, None
    q_ans = _cand_map(qa, qb).get(op)
    if q_ans is None:
        return None, None
    L = [f"I need the secret rule mapping each `a{sym}b` to its result. "
         f"I'll test operations against ALL examples and reject any that fails (Code -> Inspect -> Revise)."]
    for name, (a, b, pred, res) in reject:
        L.append(f"- Try {DESC[name]}: for {a}{sym}{b} it gives {pred}, but the example shows {res}. Reject, revise.")
    L.append(f"- Try {DESC[op]}: verify the arithmetic on EVERY example:")
    for (a, b, pred, res) in checks:
        L.append(f"    {a}{sym}{b}:  {arith(op, a, b)}   -> matches example {res} OK")
    L.append(f"All examples reproduce, so the rule is {DESC[op]}.")
    L.append(f"Now apply it to the query {qa}{sym}{qb}:")
    L.append(f"    {arith(op, qa, qb)}")
    cot = "<think>\n" + "\n".join(L) + "\n</think>\n\\boxed{" + q_ans + "}"
    return cot, q_ans

_OP_SYMS = list("/|\\{}*`>-#@&^?![]:%+<")

def gen_equation(rare=False):
    op = random.choice(RARE if rare else COMMON)
    sym = random.choice(_OP_SYMS)
    exs, seen, tries = [], set(), 0
    while len(exs) < 4 and tries < 80:
        tries += 1
        a, b = random.randint(10, 99), random.randint(10, 99)
        if (a, b) in seen:
            continue
        res = _cand_map(a, b).get(op)
        if res is None:
            return None
        seen.add((a, b)); exs.append((a, b, res))
    if len(exs) < 4:
        return None
    qa, qb = random.randint(10, 99), random.randint(10, 99)
    cot, ans = render_equation(exs, sym, qa, qb)
    if cot is None or ans != _cand_map(qa, qb).get(op):
        return None
    head = "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:"
    ex_lines = "\n".join(f"{a}{sym}{b} = {res}" for (a, b, res) in exs)
    prompt = head + "\n" + ex_lines + f"\nNow, determine the result for: {qa}{sym}{qb}"
    return {"category": "equation_numeric_" + ("guess" if rare else "deduce"),
            "prompt": prompt, "answer": ans, "rule": op, "cot": cot}

_ENC = list("!@#$%^&*()_+=[]{}|;:,.<>?/`~")

def gen_cryptarithm(reverse=False):
    syms = random.sample(_ENC, 10)
    dmap = {str(d): syms[d] for d in range(10)}
    op = random.choice(_OP_SYMS)
    enc = lambda n2: dmap[n2[0]] + dmap[n2[1]]
    def mk():
        a = f"{random.randint(0,9)}{random.randint(0,9)}"
        b = f"{random.randint(0,9)}{random.randint(0,9)}"
        L, R = enc(a), enc(b)
        return L, R, (R + L if reverse else L + R)
    exs, seen = [], set()
    while len(exs) < 4:
        L, R, out = mk()
        if (L, R) in seen:
            continue
        seen.add((L, R)); exs.append((L, R, out))
    qL, qR, q_ans = mk()
    lines = ["The operands are symbol-encoded digits. Two hypotheses: forward concat "
             "(LEFT then RIGHT) or reverse concat (RIGHT then LEFT). Test each on ALL examples."]
    wrong_rev = not reverse
    L0, R0, out0 = exs[0]
    lines.append(f"- Try {'reverse' if wrong_rev else 'forward'} concat: for {L0}{op}{R0} it gives "
                 f"{(R0+L0) if wrong_rev else (L0+R0)}, but the example shows {out0}. Reject, revise.")
    lines.append(f"- Try {'reverse' if reverse else 'forward'} concat: verify on every example:")
    for (L, R, out) in exs:
        pred = (R + L) if reverse else (L + R)
        lines.append(f"    {L}{op}{R} -> {'RIGHT then LEFT' if reverse else 'LEFT then RIGHT'} = {pred}   -> matches {out} OK")
    lines.append(f"All match, so the rule is {'reverse' if reverse else 'forward'} concatenation.")
    lines.append(f"Apply to {qL}{op}{qR}: {'RIGHT then LEFT' if reverse else 'LEFT then RIGHT'} = {q_ans}.")
    cot = "<think>\n" + "\n".join(lines) + "\n</think>\n\\boxed{" + q_ans + "}"
    head = "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:"
    ex_lines = "\n".join(f"{L}{op}{R} = {out}" for (L, R, out) in exs)
    prompt = head + "\n" + ex_lines + f"\nNow, determine the result for: {qL}{op}{qR}"
    return {"category": "cryptarithm_" + ("guess" if reverse else "deduce"),
            "prompt": prompt, "answer": q_ans,
            "rule": ("reverse" if reverse else "forward") + " concat", "cot": cot}

print('verify-revise engine ready (explicit-arithmetic kernel)')


In [ ]:
import random, json, csv, re
from collections import Counter

random.seed(SEED)
_GENS = {
    "equation_numeric_deduce": lambda: gen_equation(False),
    "equation_numeric_guess":  lambda: gen_equation(True),
    "cryptarithm_deduce":      lambda: gen_cryptarithm(False),
    "cryptarithm_guess":       lambda: gen_cryptarithm(True),
}
_BOX = re.compile(r"\\boxed\{([^{}]*)\}")

rows, idx = [], 0
for cat in CATEGORIES:
    g = _GENS[cat]; got = tries = 0; seen = set()
    while got < PER_CAT and tries < PER_CAT * 40:
        tries += 1
        r = g()
        if not r or r["prompt"] in seen:
            continue
        m = _BOX.findall(r["cot"])
        # final guard: well-formed + boxed answer matches the kernel's computed answer
        if not (m and m[-1] == r["answer"] and "</think>" in r["cot"]
                and r["cot"].find("</think>") < r["cot"].rfind("\\boxed{")):
            continue
        seen.add(r["prompt"])
        rows.append({"id": f"sc_{cat}_{idx:05d}", "category": cat, "rule": r["rule"],
                     "prompt": r["prompt"], "answer": r["answer"], "generated_cot": r["cot"]})
        idx += 1; got += 1
    print(f"{cat:26s} {got}/{PER_CAT}  (tries {tries})")

# CSV (columns match the v24 SFT loader: prompt / answer / generated_cot)
cols = ["id", "category", "rule", "prompt", "answer", "generated_cot"]
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for r in rows:
        w.writerow({c: r[c] for c in cols})

# chat JSONL (SFT/RAFT ready)
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        rec = {"id": r["id"], "category": r["category"], "answer": r["answer"],
               "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": r["prompt"] + PROMPT_SUFFIX},
                            {"role": "assistant", "content": r["generated_cot"]}]}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\nTOTAL {len(rows)} traces -> {OUT_CSV} + {OUT_JSONL}")
print("per-category:", dict(Counter(r["category"] for r in rows)))
_lens = [len(r["generated_cot"]) for r in rows]
print(f"cot chars min/med/max: {min(_lens)}/{sorted(_lens)[len(_lens)//2]}/{max(_lens)}")
print("\n--- sample cryptarithm_guess trace ---")
print(next(r for r in rows if r["category"] == "cryptarithm_guess")["generated_cot"])


## Train the 0.85 checkpoint on these traces

The output is **drop-in** for the existing notebooks (same `\boxed{}` contract, same compact system prompt).

**Option A — SFT (v24):** upload `spatialclaw_verify_revise.csv` as a Kaggle dataset and point
the v24 loader at it (it auto-detects `prompt` / `answer` / `generated_cot`):
```python
SFT_DATA_PATH = "/kaggle/input/<your-dataset>/spatialclaw_verify_revise.csv"
# warm-start from the 0.85 adapter; keep broad LoRA, LR 2e-4 -> a few hundred steps.
```
Best as a **mix**: concat with `sft_9500_even.csv` (the easy cats) so you don't forget them —
these verify-revise rows specifically lift the hard cats (cryptarithm, equation_guess).

**Option B — RAFT (v25):** add `spatialclaw_verify_revise.jsonl` to the RAFT pool; the verify-revise
text is exactly the **PERFECT_RESCUE** target for hard problems the policy fails on all rollouts.

**Why mix, not replace:** these are search-style traces (longer, many rejected hypotheses). Train on
them *alongside* the short one-shot traces for the easy cats, so the model learns to spend its
reasoning budget only where the rule is non-obvious.

> Eval-gate vs the 0.85 baseline. Expectation: the lift shows up on `cryptarithm_*` and
> `equation_numeric_guess` (the categories where one-shot guessing fails) — that's the SpatialClaw
> "action interface" effect: the CoT is now a verify-revise search, not a single guess.
